# Mu-Logarit - group4 (D33-D42, 9 dạng) - FULL90 - ZERO-SHOT - Qwen3-4B

Dùng đúng PROMPT_TEMPLATE tiếng Việt, cách chấm điểm, và các tham số sinh (`temperature=0.6, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2, max_new_tokens=9216, enable_thinking=True`) giống hệt bản Qwen3-4B zero-shot đã dùng cho bộ Số phức - không đổi chữ nào ngoài tên model (giữ nguyên) và nguồn dữ liệu.

**CẦN UPLOAD** `plan_solve_prompts_mulogarit_merged.json` lên Kaggle, đường dẫn `/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json`.


In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)


In [ ]:
import os, json, re, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams


## Dạng D33 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d33_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D33']
print('So cau:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D33, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D34 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d34_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D34']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D34, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D35 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d35_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D35']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D35, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D37 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d37_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D37']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D37, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D38 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d38_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D38']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D38, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D39 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d39_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D39']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D39, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D40 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d40_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D40']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D40, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D41 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d41_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D41']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D41, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dạng D42 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/mulogarit-full/plan_solve_prompts_mulogarit_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d42_zeroshot_full90_Qwen3-4B.csv'

MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D42']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dau tien cua
# notebook nay. Nap lai LLM(...) nhieu lan hay gay loi GPU het bo nho.
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after)

    rows.append({
        'STT': r['STT'], 'ma_dang': r['ma_dang'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D42, Qwen3-4B):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])
